## Definitions

Let subscripts denote nodal or stencil and superscripts denote edge quantites. For an edge $e^i$, let

- $\mathbf{t}^i$ be the tangent vector.
- $\{\mathbf{d}_1^i, \mathbf{d}_2^i, \mathbf{t}^i\}$ be the orthonormal reference frame.
- $\{\mathbf{m}_1^i, \mathbf{m}_2^i, \mathbf{t}^i\}$ be the orthonormal material frame and the rotation of reference frame by a signed angle $\theta^i$ along $\mathbf{t}^i$.

## Formulation

For a 11 node stencil, let

- $\mathbf{q}=[x_0, y_0, z_0, \theta^0, x_1, y_1, z_1, \theta^1, x_2, y_2, z_2]^T$ be the state vector.
- $\mathbf{aux}=[\mathbf{t}^0_{\textrm{old}}, \mathbf{t}^1_{\textrm{old}}, \mathbf{d}_{1,\textrm{old}}^0, \mathbf{d}_{1,\textrm{old}}^1, \beta_\textrm{old}]^T$ be the auxiliary vector.
- $\boldsymbol{\epsilon}=[\epsilon^0, \epsilon^1, \kappa_1, \kappa_2, \tau]^T$ be the strain vector.
- $E=f(\boldsymbol{\epsilon})$ be the energy.

### Frame Reconstruction

First, we calcualte the $\mathbf{d}_{1,\textrm{new}}^i$ from $\mathbf{aux}$

$$
\mathbf{d}_{1,\textrm{new}}^i=\textrm{ParallelTransport}(\mathbf{d}_{1,\textrm{old}}^i,\mathbf{t}_{\textrm{old}}^i, \mathbf{t}_{\textrm{new}}^i)
$$

With $\mathbf{t}_{\textrm{new}}^i$ and $\mathbf{d}_{1,\textrm{new}}^i$ we construct the **new** orthonormal reference frame $\{\mathbf{d}_1^i, \mathbf{d}_2^i, \mathbf{t}^i\}$ as

$$
\mathbf{d}_{2,\textrm{new}}^i=\mathbf{t}^i_{\textrm{new}}\times \mathbf{d}_{1,\textrm{new}}^i
$$

With $\{\mathbf{d}_1^i, \mathbf{d}_2^i, \mathbf{t}^i\}$ and $\theta^i$, we compute the material frame $\{\mathbf{m}_1^i, \mathbf{m}_2^i, \mathbf{t}^i\}$

$$
\begin{align*}
m_1^i&=\cos(\theta^i)\mathbf{d}_1^i+\sin(\theta^i)\mathbf{d}_2^i \\
m_2^i&=-\sin(\theta^i)\mathbf{d}_1^i+\cos(\theta^i)\mathbf{d}_2^i
\end{align*}
$$

### Strains

#### Stretching Strain 

The scalar stretching strain is the ratio of elongation / compression of an edge:

$$
\epsilon^i=\frac{\|\mathbf{e}^i\|}{\|\mathbf{\bar e}^i\|}-1
$$

#### Bending Strains

The discrete curvature binormal $(\kappa\mathbf{b})$ at node $i$ represents the integrated curvature between edges $\mathbf{e}^0$ and $\mathbf{e}^1$:

$$
(\kappa\mathbf{b})=\frac{2(\mathbf{t^0}\times \mathbf{t^1})}{1+\mathbf{t^0}\cdot \mathbf{t^1}}
$$

The scalar bending strains are the projections of this binormal onto the averaged material directors:

$$
\begin{align*}
\kappa_1&=\frac{1}{2}(\kappa\mathbf{b})\cdot(\mathbf{m}_2^0+\mathbf{m}_2^1) \\
\kappa_2&=-\frac{1}{2}(\kappa\mathbf{b})\cdot(\mathbf{m}_1^0+\mathbf{m}_1^1) 
\end{align*}
$$

#### Twisting Strain

The twisting strain $\tau$ accounts for the relative rotation of material frames plus the geometric holonomy:

$$
\tau=\theta^1-\theta^0+\beta
$$

Where $\beta$ is the angle of parallel transport between reference frames. For $\mathbf{t}^0 \cdot \mathbf{t}^1 > -1$:

$$\beta = \operatorname{atan2}\left( (\mathbf{d}_{1,\textrm{new}}^0 \times \mathbf{d}_{1,\textrm{new}}^1) \cdot \mathbf{\bar{t}}, \,\, \mathbf{d}_{1,\textrm{new}}^0 \cdot \mathbf{d}_{1,\textrm{new}}^1 \right), \quad \mathbf{\bar{t}} = \frac{\mathbf{t}^0 \times \mathbf{t}^1}{\|\mathbf{t}^0 \times \mathbf{t}^1\|}$$

In [1]:
"""import symengine as sp


# symengine lacks norm
def norm(u):
    return sp.sqrt(u.dot(u))


def parallel_transport(u, t1, t2):
    b = t1.cross(t2)
    d = t1.dot(t2)
    denom = 1.0 + d
    b_cross_u = b.cross(u)
    return u + b_cross_u + b.cross(b_cross_u) / denom


def signed_angle(u, v, n):
    w = u.cross(v)
    return sp.atan2(w.dot(n), u.dot(v))


def rotate_axis_angle(u, v, theta):
    c = sp.cos(theta)
    s = sp.sin(theta)
    return c * u + s * v.cross(u) + v.dot(u) * (1.0 - c) * v


def material_frame(d1, d2, theta):
    c = sp.cos(theta)
    s = sp.sin(theta)
    return c * d1 + s * d2, -s * d1 + c * d2


# Dofs: [x0, y0, z0, theta0, x1, y1, z1, theta1, x2, y2, z2]
x0 = sp.Matrix([sp.Symbol(f"x_{i}_0") for i in range(3)])
x1 = sp.Matrix([sp.Symbol(f"x_{i}_1") for i in range(3)])
x2 = sp.Matrix([sp.Symbol(f"x_{i}_2") for i in range(3)])
theta = sp.Matrix([sp.Symbol("theta_0"), sp.Symbol("theta_1")])

# Parameters
l_k = sp.Matrix([sp.Symbol("len_0"), sp.Symbol("len_1")])
t0_old = sp.Matrix([sp.Symbol(f"t0_old_{i}") for i in range(3)])
t1_old = sp.Matrix([sp.Symbol(f"t1_old_{i}") for i in range(3)])
d10_old = sp.Matrix([sp.Symbol(f"d10_old_{i}") for i in range(3)])
d11_old = sp.Matrix([sp.Symbol(f"d11_old_{i}") for i in range(3)])
ref_twist_old = sp.Symbol("beta_old")

e0 = x1 - x0
e1 = x2 - x1
n_e0 = norm(e0)
n_e1 = norm(e1)
t0 = e0 / n_e0
t1 = e1 / n_e1

# Transport
d10_new = parallel_transport(d10_old, t0_old, t0)
d11_new = parallel_transport(d11_old, t1_old, t1)
d20_new = t0.cross(d10_new)
d21_new = t1.cross(d11_new)

# Material Frames
m1e, m2e = material_frame(d10_new, d20_new, theta[0])
m1f, m2f = material_frame(d11_new, d21_new, theta[1])

# Stretch (Epsilon)
eps0 = n_e0 / l_k[0] - 1.0
eps1 = n_e1 / l_k[1] - 1.0

# Curvature (Kappa)
kb = (2.0 * t0.cross(t1)) / (1.0 + t0.dot(t1))
kappa1 = 0.5 * kb.dot(m2e + m2f)
kappa2 = -0.5 * kb.dot(m1e + m1f)

# Twist (Tau)
d1_transport = parallel_transport(d10_new, t0, t1)
d1_pred = rotate_axis_angle(d1_transport, t1, ref_twist_old)
delta_phi = signed_angle(d1_pred, d11_new, t1)
tau = theta[1] - theta[0] + ref_twist_old + delta_phi

# DER Energy (Example/base)
K1, K2, K3, K4 = sp.symbols("K_epsilon K_kappa1 K_kappa2 K_tau")
E = 0.5 * (K1 * eps0**2 + K1 * eps1**2 + K2 * kappa1**2 + K3 * kappa2**2 + K4 * tau**2)

# Derive analytical gradients/hessians
dofs = list(x0) + [theta[0]] + list(x1) + [theta[1]] + list(x2)
strains = [eps0, eps1, kappa1, kappa2, tau]

a = sp.diff(E, dofs[0])"""

'import symengine as sp\n\n\n# symengine lacks norm\ndef norm(u):\n    return sp.sqrt(u.dot(u))\n\n\ndef parallel_transport(u, t1, t2):\n    b = t1.cross(t2)\n    d = t1.dot(t2)\n    denom = 1.0 + d\n    b_cross_u = b.cross(u)\n    return u + b_cross_u + b.cross(b_cross_u) / denom\n\n\ndef signed_angle(u, v, n):\n    w = u.cross(v)\n    return sp.atan2(w.dot(n), u.dot(v))\n\n\ndef rotate_axis_angle(u, v, theta):\n    c = sp.cos(theta)\n    s = sp.sin(theta)\n    return c * u + s * v.cross(u) + v.dot(u) * (1.0 - c) * v\n\n\ndef material_frame(d1, d2, theta):\n    c = sp.cos(theta)\n    s = sp.sin(theta)\n    return c * d1 + s * d2, -s * d1 + c * d2\n\n\n# Dofs: [x0, y0, z0, theta0, x1, y1, z1, theta1, x2, y2, z2]\nx0 = sp.Matrix([sp.Symbol(f"x_{i}_0") for i in range(3)])\nx1 = sp.Matrix([sp.Symbol(f"x_{i}_1") for i in range(3)])\nx2 = sp.Matrix([sp.Symbol(f"x_{i}_2") for i in range(3)])\ntheta = sp.Matrix([sp.Symbol("theta_0"), sp.Symbol("theta_1")])\n\n# Parameters\nl_k = sp.Matrix([sp

In [ ]:
import warp as wp
import numpy as np


@wp.kernel
def epsilon_der(
    nodes: wp.array(dtype=wp.vec3),
    l_ks: wp.array(dtype=float),
    ks: wp.array(dtype=float),
    energy: wp.array(dtype=float),
):
    idx = wp.tid()
    n0, n1 = nodes[idx], nodes[idx + 1]
    e_len = wp.length(n1 - n0)
    eps = e_len / l_ks[idx] - 1.0
    e_val = 0.5 * ks[idx] * eps * eps
    wp.atomic_add(energy, 0, e_val)


@wp.kernel
def grad_epsilon_der(
    nodes: wp.array(dtype=wp.vec3),
    l_ks: wp.array(dtype=float),
    ks: wp.array(dtype=float),
    F: wp.array(dtype=float),
):
    idx = wp.tid()
    n0, n1 = nodes[idx], nodes[idx + 1]
    inv_lk = 1.0 / l_ks[idx]
    e = n1 - n0
    e_len = wp.length(e)
    eps = e_len * inv_lk - 1.0
    tangent = e / e_len
    deps = tangent * inv_lk
    k = ks[idx]
    dF = k * eps * deps

    # F = [-dF, dF]
    base = idx * 4  # skip a node
    wp.atomic_add(F, base + 0, -dF[0])
    wp.atomic_add(F, base + 1, -dF[1])
    wp.atomic_add(F, base + 2, -dF[2])
    wp.atomic_add(F, base + 4, dF[0])  # skip theta
    wp.atomic_add(F, base + 5, dF[1])
    wp.atomic_add(F, base + 6, dF[2])


@wp.kernel
def hess_epsilon_der(
    nodes: wp.array(dtype=wp.vec3),
    l_ks: wp.array(dtype=float),
    ks: wp.array(dtype=float),
    H: wp.array2d(dtype=float),
):
    idx = wp.tid()
    n0, n1 = nodes[idx], nodes[idx + 1]
    inv_lk = 1.0 / l_ks[idx]
    e = n1 - n0
    e_len = wp.length(e)
    eps = e_len * inv_lk - 1.0
    inv_e = 1.0 / e_len
    tangent = e * inv_e
    deps = tangent * inv_lk
    tt_T = wp.outer(tangent, tangent)
    I3 = wp.mat33(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)
    ddeps = (I3 - tt_T) * inv_lk * inv_e
    k = ks[idx]
    dJ = k * wp.outer(deps, deps) + k * eps * ddeps

    # H = [[dJ, -dJ], [-dJ, dJ]]
    base = idx * 4  # skip a node
    for r in range(3):
        for c in range(3):
            val = dJ[r, c]
            wp.atomic_add(H, base + r + 0, base + c + 0, val)  # [0,0]
            wp.atomic_add(H, base + r + 0, base + c + 4, -val)  # [0, 1]
            wp.atomic_add(H, base + r + 4, base + c + 0, -val)  # [1, 0]
            wp.atomic_add(H, base + r + 4, base + c + 4, val)  # [1, 1]


def run_benchmark(num_edges=100000, iterations=1000):
    wp.init()
    device = "cuda" if wp.is_cuda_available() else "cpu"
    num_nodes = num_edges + 1

    print(f"--- Benchmarking on {device} with {num_edges} edges ---")

    # 1. Synthetic Data Generation
    # Random positions in a [0, 10] box
    nodes_np = np.random.rand(num_nodes, 3).astype(np.float32) * 10.0
    l_ks_np = np.random.rand(num_edges).astype(np.float32) + 0.5
    ks_np = np.random.rand(num_edges).astype(np.float32) * 100.0

    # Upload to GPU
    nodes = wp.array(nodes_np, dtype=wp.vec3, device=device)
    l_ks = wp.array(l_ks_np, dtype=float, device=device)
    ks = wp.array(ks_np, dtype=float, device=device)

    # Pre-allocate outputs
    F = wp.zeros(num_nodes * 4, dtype=float, device=device)
    # H = wp.zeros((num_nodes * 4, num_nodes * 4), dtype=float, device=device)

    # 2. Warm-up (Compiles the kernel)
    print("Compiling and warming up...")
    wp.launch(
        grad_epsilon_der,
        dim=num_edges,
        inputs=[nodes, l_ks, ks],
        outputs=[F],
        device=device,
    )
    """wp.launch(
        hess_epsilon_der,
        dim=num_edges,
        inputs=[nodes, l_ks, ks],
        outputs=[H],
        device=device,
    )"""
    wp.synchronize()

    # 3. Timing Gradient Kernel
    print(f"Running Gradient Kernel for {iterations} iterations...")
    with wp.ScopedTimer("Gradient Kernel", active=True):
        for _ in range(iterations):
            wp.launch(
                grad_epsilon_der,
                dim=num_edges,
                inputs=[nodes, l_ks, ks],
                outputs=[F],
                device=device,
            )
        wp.synchronize()

    """print(f"Running Hessian Kernel for {iterations} iterations...")
    with wp.ScopedTimer("Gradient Kernel", active=True):
        for _ in range(iterations):
            wp.launch(
                hess_epsilon_der,
                dim=num_edges,
                inputs=[nodes, l_ks, ks],
                outputs=[H],
                device=device,
            )
        wp.synchronize()"""


run_benchmark()

Warp 1.11.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "AMD64 Family 26 Model 68 Stepping 0, AuthenticAMD"
     "cuda:0"   : "NVIDIA GeForce RTX 5070 Ti" (16 GiB, sm_120, mempool enabled)
   Kernel cache:
     \\?\C:\Users\12345\AppData\Local\NVIDIA\warp\Cache\1.11.1
--- Benchmarking on cuda with 100000 edges ---
Compiling and warming up...
Module __main__ 5149fb3 load on device 'cuda:0' took 493.60 ms  (compiled)
Running Gradient Kernel for 1000 iterations...
Gradient Kernel took 10.68 ms


In [ ]:
def run_fdm_check():
    print("\n--- Running Finite Difference Check ---")
    wp.init()
    device = "cuda" if wp.is_cuda_available() else "cpu"
    
    # Small setup for verification
    num_edges = 2
    num_nodes = num_edges + 1
    # 4 DOFs per node (x,y,z,theta), though theta is unused here
    dof_per_node = 4 
    total_dofs = num_nodes * dof_per_node

    # Initialize Data
    nodes_np = np.array([[0,0,0], [1,0,0], [2,1,0]], dtype=np.float32)
    l_ks_np = np.array([1.0, 1.0], dtype=np.float32) # Rest lengths
    ks_np = np.array([100.0, 100.0], dtype=np.float32) # Stiffness

    nodes = wp.array(nodes_np, dtype=wp.vec3, device=device)
    l_ks = wp.array(l_ks_np, dtype=float, device=device)
    ks = wp.array(ks_np, dtype=float, device=device)
    
    # Buffers
    F_analytic = wp.zeros(total_dofs, dtype=float, device=device)
    H_analytic = wp.zeros((total_dofs, total_dofs), dtype=float, device=device)
    energy_out = wp.zeros(1, dtype=float, device=device)

    # 1. Compute Analytic Results
    wp.launch(grad_epsilon_der, dim=num_edges, inputs=[nodes, l_ks, ks, F_analytic], device=device)
    wp.launch(hess_epsilon_der, dim=num_edges, inputs=[nodes, l_ks, ks, H_analytic], device=device)
    
    F_warp = F_analytic.numpy()
    H_warp = H_analytic.numpy()

    # 2. Finite Difference Gradient Check
    # Force = -Grad(Energy). FDM calculates Grad(Energy). 
    # Check: F_warp approx -FDM_Grad
    
    eps = 1e-4
    F_fdm = np.zeros_like(F_warp)
    
    # Flatten nodes for easier perturbation logic
    nodes_flat = nodes_np.flatten() 
    
    for i in range(num_nodes):
        for axis in range(3): # x, y, z
            dof_idx = i * 4 + axis
            
            # Perturb +
            orig = nodes_np[i, axis]
            nodes_np[i, axis] = orig + eps
            wp.copy(nodes, wp.array(nodes_np, dtype=wp.vec3, device=device))
            energy_out.zero_()
            wp.launch(compute_energy, dim=num_edges, inputs=[nodes, l_ks, ks, energy_out], device=device)
            e_plus = energy_out.numpy()[0]
            
            # Perturb -
            nodes_np[i, axis] = orig - eps
            wp.copy(nodes, wp.array(nodes_np, dtype=wp.vec3, device=device))
            energy_out.zero_()
            wp.launch(compute_energy, dim=num_edges, inputs=[nodes, l_ks, ks, energy_out], device=device)
            e_minus = energy_out.numpy()[0]
            
            # Restore
            nodes_np[i, axis] = orig
            
            # Central Difference
            F_fdm[dof_idx] = (e_plus - e_minus) / (2.0 * eps)

    # Compare Forces (-Grad vs FDM Grad)
    # The kernel computes Forces (negative gradient), so we compare F_warp vs -F_fdm
    print(f"Gradient Check (Analytic Force vs -FDM Grad):")
    diff_grad = np.max(np.abs(F_warp - (-F_fdm)))
    print(f"Max Diff: {diff_grad:.6e}")
    if diff_grad < 1e-3:
        print(">> Gradient Verified!")
    else:
        print(">> Gradient Mismatch!")

    # 3. Finite Difference Hessian Check
    # Hessian = Jacobian of Gradient.
    # We will perturb nodes and compute finite differences of the ANALYTIC GRADIENT (Forces)
    # Since F = -Grad E, Jacobian(F) = -Hessian.
    # So H_fdm approx - (F(x+h) - F(x-h)) / 2h
    
    H_fdm = np.zeros_like(H_warp)
    temp_F = wp.zeros_like(F_analytic)
    
    for i in range(num_nodes):
        for axis in range(3):
            col_idx = i * 4 + axis
            
            # Perturb +
            orig = nodes_np[i, axis]
            nodes_np[i, axis] = orig + eps
            wp.copy(nodes, wp.array(nodes_np, dtype=wp.vec3, device=device))
            temp_F.zero_()
            wp.launch(grad_epsilon_der, dim=num_edges, inputs=[nodes, l_ks, ks, temp_F], device=device)
            F_plus = temp_F.numpy()
            
            # Perturb -
            nodes_np[i, axis] = orig - eps
            wp.copy(nodes, wp.array(nodes_np, dtype=wp.vec3, device=device))
            temp_F.zero_()
            wp.launch(grad_epsilon_der, dim=num_edges, inputs=[nodes, l_ks, ks, temp_F], device=device)
            F_minus = temp_F.numpy()
            
            # Restore
            nodes_np[i, axis] = orig
            
            # Central Diff of Force
            # Stiffness Matrix K = - dF/dx
            # But Hessian usually refers to d2E/dx2.
            # Since F = -dE/dx, then dF/dx = -d2E/dx2 = -Hessian
            # So Hessian = - (dF/dx)
            col_val = -(F_plus - F_minus) / (2.0 * eps)
            H_fdm[:, col_idx] = col_val

    # Mask out the 4th row/col (theta) as it is unused and zero
    mask = np.ones_like(H_warp, dtype=bool)
    for k in range(num_nodes):
        mask[k*4 + 3, :] = False
        mask[:, k*4 + 3] = False
    
    diff_hess = np.max(np.abs(H_warp[mask] - H_fdm[mask]))
    print(f"\nHessian Check (Analytic vs FDM of Analytic Force):")
    print(f"Max Diff: {diff_hess:.6e}")
    if diff_hess < 1e-2: # Hessians are sensitive to 2nd order numerical noise
        print(">> Hessian Verified!")
    else:
        print(">> Hessian Mismatch!")

In [8]:
@wp.kernel
def grad_kappa_der(
    nodes: wp.array(dtype=wp.vec3),
    m1s: wp.array(dtype=wp.vec3),
    m2s: wp.array(dtype=wp.vec3),
    ks: wp.array(dtype=float),
    F: wp.array(dtype=float),
):
    idx = wp.tid()
    n0, n1, n2 = nodes[idx], nodes[idx + 1], nodes[idx + 2]
    m1e, m2e = m1s[idx * 2], m2s[idx * 2]
    m1f, m2f = m1s[idx * 2 + 1], m2s[idx * 2 + 1]

    ee = n1 - n0
    ef = n2 - n1
    inv_ee = 1.0 / wp.length(ee)
    inv_ef = 1.0 / wp.length(ef)
    te = ee * inv_ee
    tf = ef * inv_ef

    inv_denom = 1.0 / (1.0 + wp.dot(te, tf))  # TODO: add eps?
    kb = 2.0 * wp.cross(te, tf) * inv_denom

    kappa1 = 0.5 * wp.dot(kb, m2e + m2f)
    kappa2 = -0.5 * wp.dot(kb, m1e + m1f)

    k1_factor = ks[idx] * kappa1
    k2_factor = ks[idx + 1] * kappa2
    tilde_t = (te + tf) * inv_denom
    tilde_d1 = (m1e + m1f) * inv_denom
    tidle_d2 = (m2e + m2f) * inv_denom
    de = k1_factor * inv_ee * (-kappa1 * tilde_t + wp.cross(tf, tidle_d2))
    de += k2_factor * inv_ee * (-kappa2 * tilde_t - wp.cross(tf, tilde_d1))
    df = k1_factor * inv_ef * (-kappa1 * tilde_t + wp.cross(tf, tidle_d2))
    df += k2_factor * inv_ee * (-kappa2 * tilde_t - wp.cross(tf, tilde_d1))
    dthetae = k1_factor * -0.5 * wp.dot(kb, m1e)
    dthetae += k2_factor * -0.5 * wp.dot(kb, m2e)
    dthetaf = k1_factor * -0.5 * wp.dot(kb, m1f)
    dthetaf += k2_factor * -0.5 * wp.dot(kb, m2f)

    base = idx * 4  # skip a node
    wp.atomic_add(F, base + 0, -de[0])
    wp.atomic_add(F, base + 1, -de[1])
    wp.atomic_add(F, base + 2, -de[2])
    wp.atomic_add(F, base + 3, dthetae)
    wp.atomic_add(F, base + 4, de[0] - df[0])
    wp.atomic_add(F, base + 5, de[1] - df[1])
    wp.atomic_add(F, base + 6, de[2] - df[2])
    wp.atomic_add(F, base + 7, dthetaf)
    wp.atomic_add(F, base + 8, df[0])
    wp.atomic_add(F, base + 9, df[1])
    wp.atomic_add(F, base + 10, df[2])


@wp.kernel
def hess_kappa_der(
    nodes: wp.array(dtype=wp.vec3),
    m1s: wp.array(dtype=wp.vec3),
    m2s: wp.array(dtype=wp.vec3),
    ks: wp.array(dtype=float),
    F: wp.array(dtype=float),
):
    idx = wp.tid()
    n0, n1, n2 = nodes[idx], nodes[idx + 1], nodes[idx + 2]
    m1e, m2e = m1s[idx * 2], m2s[idx * 2]
    m1f, m2f = m1s[idx * 2 + 1], m2s[idx * 2 + 1]

    ee = n1 - n0
    ef = n2 - n1
    inv_ee = 1.0 / wp.length(ee)
    inv_ef = 1.0 / wp.length(ef)
    te = ee * inv_ee
    tf = ef * inv_ef

    inv_denom = 1.0 / (1.0 + wp.dot(te, tf))  # TODO: add eps?
    kb = 2.0 * wp.cross(te, tf) * inv_denom

    kappa1 = 0.5 * wp.dot(kb, m2e + m2f)
    kappa2 = -0.5 * wp.dot(kb, m1e + m1f)

    tilde_t = (te + tf) * inv_denom
    # tilde_d1 = (m1e + m1f) * inv_denom
    tilde_d2 = (m2e + m2f) * inv_denom

    tt_o_tt = wp.outer(tilde_t, tilde_t)
    tf_c_d2t_o_tt = wp.outer(wp.cross(tf, tilde_d2), tilde_t)
    tt_o_tf_c_d2t = wp.transpose(tf_c_d2t_o_tt)
    kb_o_d2e = wp.outer(kb, m2e)
    d2e_o_kb = wp.transpose(kb_o_d2e)

    I3 = wp.mat33(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)

    D2kappa1De2 = (
        inv_ee * inv_ee * (2.0 * kappa1 * tt_o_tt - tf_c_d2t_o_tt - tt_o_tf_c_d2t)
        - kappa1 * inv_denom * inv_ee * inv_ee * (I3 - wp.outer(te, te))
        + 0.25 * inv_ee * inv_ee * (kb_o_d2e + d2e_o_kb)
    )

    te_c_d2t_o_tt = wp.outer(wp.cross(te, tilde_d2), tilde_t)
    tt_o_te_c_d2t = wp.transpose(te_c_d2t_o_tt)
    kb_o_d2f = wp.outer(kb, m2f)
    d2f_o_kb = wp.transpose(kb_o_d2f)
    tf_o_tf = wp.outer(tf, tf)

    D2kappa1Df2 = (
        inv_ef * inv_ef * (2.0 * kappa1 * tt_o_tt + te_c_d2t_o_tt + tt_o_te_c_d2t)
        - kappa1 * inv_denom * inv_ee * inv_ee * (I3 - tf_o_tf)
        + 0.25 * inv_ef * inv_ef * (kb_o_d2f + d2f_o_kb)
    )

    D2kappa1DeDf = -kappa1 * inv_denom * inv_ee * inv_ef * (I3 + wp.outer(te, tf))
    +1.0 * inv_ee * inv_ef * (
        2.0 * kappa1 * tt_o_tt
        - tf_c_d2t_o_tt
        + tt_o_te_c_d2t
    #    - crossMat(tilde_d2)  # todo cross mat
    )
    D2kappa1DfDe = wp.transpose(D2kappa1DeDf)


def run_benchmark(num_edges=100000, iterations=1000):
    wp.init()
    device = "cuda" if wp.is_cuda_available() else "cpu"
    num_nodes = num_edges + 1

    print(f"--- Benchmarking on {device} with {num_edges} edges ---")

    # 1. Synthetic Data Generation
    # Random positions in a [0, 10] box
    nodes_np = np.random.rand(num_nodes, 3).astype(np.float32) * 10.0
    m1s_np = np.random.rand(num_edges, 3).astype(np.float32) * 10.0
    m2s_np = np.random.rand(num_edges, 3).astype(np.float32) * 10.0
    ks_np = np.random.rand(num_edges).astype(np.float32) * 100.0

    # Upload to GPU
    nodes = wp.array(nodes_np, dtype=wp.vec3, device=device)
    m1s = wp.array(m1s_np, dtype=wp.vec3, device=device)
    m2s = wp.array(m2s_np, dtype=wp.vec3, device=device)
    ks = wp.array(ks_np, dtype=float, device=device)

    # Pre-allocate outputs
    F = wp.zeros(num_nodes * 4, dtype=float, device=device)
    # H = wp.zeros((num_nodes * 3, num_nodes * 3), dtype=float, device=device)

    # 2. Warm-up (Compiles the kernel)
    print("Compiling and warming up...")
    wp.launch(
        grad_kappa_der,
        dim=num_edges,
        inputs=[nodes, m1s, m2s, ks],
        outputs=[F],
        device=device,
    )
    """wp.launch(
        hess_epsilon_der,
        dim=num_edges,
        inputs=[nodes, l_ks, ks],
        outputs=[H],
        device=device,
    )"""
    wp.synchronize()

    # 3. Timing Gradient Kernel
    print(f"Running Gradient Kernel for {iterations} iterations...")
    with wp.ScopedTimer("Gradient Kernel", active=True):
        for _ in range(iterations):
            wp.launch(
                grad_kappa_der,
                dim=num_edges,
                inputs=[nodes, m1s, m2s, ks],
                outputs=[F],
                device=device,
            )
        wp.synchronize()

    """print(f"Running Hessian Kernel for {iterations} iterations...")
    with wp.ScopedTimer("Gradient Kernel", active=True):
        for _ in range(iterations):
            wp.launch(
                hess_epsilon_der,
                dim=num_edges,
                inputs=[nodes, l_ks, ks],
                outputs=[H],
                device=device,
            )
        wp.synchronize"""


run_benchmark()

--- Benchmarking on cuda with 100000 edges ---
Compiling and warming up...
Running Gradient Kernel for 1000 iterations...
Gradient Kernel took 20.86 ms
